In [5]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [6]:
load_dotenv()

llm = ChatOpenAI()

In [8]:
class JokeState(TypedDict):

    topic:str
    joke:str
    explanation:str


In [9]:
def generate_joke(state: JokeState):

    prompt = f"generate a joke on the given topic {state['topic']}"
    response = llm.invoke(prompt).content

    return {'joke': response}

In [11]:
def generate_explanation(state:JokeState):

    prompt = f"write an explanation for the joke - {state['joke']}"
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [13]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)


In [17]:
config1 = {"configurable": {"thread_id":"2"}}
workflow.invoke({'topic':'pasta'}, config=config1)

{'topic': 'pasta',
 'joke': 'Why did the pasta go to the party? \nBecause it heard it was going to be a pasta-tively great time!',
 'explanation': 'This joke plays on the word "pasta-tively" which is a play on the words "positive" and "pasta". The pasta decided to go to the party because it heard that it was going to be a really fun time, and the play on words adds a humorous touch to the punchline. Overall, the joke is meant to be light-hearted and silly.'}

In [18]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the pasta go to the party? \nBecause it heard it was going to be a pasta-tively great time!', 'explanation': 'This joke plays on the word "pasta-tively" which is a play on the words "positive" and "pasta". The pasta decided to go to the party because it heard that it was going to be a really fun time, and the play on words adds a humorous touch to the punchline. Overall, the joke is meant to be light-hearted and silly.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12eb2a-5f81-66e3-8002-29876fb1ca86'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-02T16:40:24.875184+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f12eb2a-49bc-6136-8001-4a6a4456a4a0'}}, tasks=(), interrupts=())

In [19]:
list(workflow.get_state(config1))

[{'topic': 'pasta',
  'joke': 'Why did the pasta go to the party? \nBecause it heard it was going to be a pasta-tively great time!',
  'explanation': 'This joke plays on the word "pasta-tively" which is a play on the words "positive" and "pasta". The pasta decided to go to the party because it heard that it was going to be a really fun time, and the play on words adds a humorous touch to the punchline. Overall, the joke is meant to be light-hearted and silly.'},
 (),
 {'configurable': {'thread_id': '2',
   'checkpoint_ns': '',
   'checkpoint_id': '1f12eb2a-5f81-66e3-8002-29876fb1ca86'}},
 {'source': 'loop', 'step': 2, 'parents': {}},
 '2026-04-02T16:40:24.875184+00:00',
 {'configurable': {'thread_id': '2',
   'checkpoint_ns': '',
   'checkpoint_id': '1f12eb2a-49bc-6136-8001-4a6a4456a4a0'}},
 (),
 ()]